# COMP5318 Assignment 1: Rice Classification

##### Group number: 184
##### Student 1 SID: ...
##### Student 2 SID: ...  

## **1. Data Pre-processing**

In [1]:
# Import all libraries
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Load the rice dataset: rice-final2.csv
# Missing values are recorded as '?' and are read in as NaN so that they can be imputed later.
# The number of features and examples is never hard-coded: the class variable is assumed to be
# the last column and every preceding column is treated as a feature. This keeps the pipeline
# usable on any dataset with the same format (e.g. the unknown dataset used for marking).

DATA_PATH = 'rice-final2.csv'

def load_dataset(file_path):
    """Read a CSV dataset and split it into the raw feature matrix and the raw class column.

    Arguments:
        file_path: path to a CSV file whose last column holds the class label
                   and whose missing values are recorded as '?'

    Returns:
        X_raw: DataFrame of shape (n_examples, n_features)
        y_raw: Series of shape (n_examples,) holding the class labels as strings
    """
    data_frame = pd.read_csv(file_path, na_values='?')
    X_raw = data_frame.iloc[:, :-1]   # all columns except the last one are features
    y_raw = data_frame.iloc[:, -1]    # the last column is the class variable
    return X_raw, y_raw

X_raw, y_raw = load_dataset(DATA_PATH)
print("Number of examples:", X_raw.shape[0])
print("Number of features:", X_raw.shape[1])

Number of examples: 209
Number of features: 6


In [4]:
# Pre-process dataset
# Three steps are required:
#   1. fill in the missing feature values with the mean of their column (SimpleImputer)
#   2. normalise every feature to the range [0, 1] (MinMaxScaler)
#   3. map the class labels class1 -> 0 and class2 -> 1

CLASS_MAPPING = {'class1': 0, 'class2': 1}

def preprocess(X_raw, y_raw):
    """Impute missing values, min-max normalise the features and encode the class labels.

    Arguments:
        X_raw: DataFrame of raw feature values (may contain NaN)
        y_raw: Series of raw class labels ('class1' / 'class2')

    Returns:
        X: numpy array of shape (n_examples, n_features), all values in [0, 1]
        y: numpy array of shape (n_examples,) containing 0s and 1s
    """
    # Force every feature column to be numeric; anything unparsable becomes NaN so that
    # it is handled by the imputer together with the '?' entries.
    X_numeric = X_raw.apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)

    # 1. Replace the missing values with the mean value of the corresponding column
    imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
    X_imputed = imputer.fit_transform(X_numeric)

    # 2. Normalise each feature to [0, 1]
    scaler = MinMaxScaler(feature_range=(0, 1))
    X = scaler.fit_transform(X_imputed)

    # 3. Change the class values: class1 -> 0, class2 -> 1
    y = y_raw.astype(str).str.strip().map(CLASS_MAPPING).to_numpy(dtype=int)

    return X, y

X, y = preprocess(X_raw, y_raw)
print("Pre-processed data shape:", X.shape)
print("Class distribution (class 0, class 1):", np.bincount(y))

Pre-processed data shape: (209, 6)
Class distribution (class 0, class 1): [ 88 121]


In [5]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            

print_data(X, y)

0.0621,0.4999,0.5410,0.2079,0.2594,0.0613,0
0.8073,0.7474,0.6721,0.2634,0.2038,0.0586,0
0.3105,0.6030,0.4187,0.0000,0.0000,0.0900,0
0.3105,0.5618,0.6148,0.3604,0.0000,0.0950,0
0.1863,0.8144,0.6230,0.4990,0.4539,0.1597,1
0.1863,0.6039,0.4754,0.1525,0.1000,0.0655,1
0.6832,0.7114,0.6230,0.0000,0.0000,0.0877,1
0.5589,0.5258,0.6230,0.5129,0.0000,0.0869,0
0.1242,0.4639,0.5574,0.5822,0.0000,0.1009,1
0.2484,0.5722,0.5902,0.6515,0.3835,0.0979,0


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [6]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [7]:
# Logistic Regression
# No parameter tuning in Part 1, so the classifier is evaluated directly with 10-fold
# stratified cross-validation on the whole pre-processed dataset.

def logregClassifier(X, y):
    """Return the average 10-fold stratified cross-validation accuracy of Logistic Regression."""
    classifier = LogisticRegression(random_state=0)
    scores = cross_val_score(classifier, X, y, cv=cvKFold, scoring='accuracy')
    return scores.mean()

logreg_accuracy = logregClassifier(X, y)

In [8]:
# Naïve Bayes
# The features are continuous after min-max normalisation, so the Gaussian variant is used.

def nbClassifier(X, y):
    """Return the average 10-fold stratified cross-validation accuracy of Gaussian Naive Bayes."""
    classifier = GaussianNB()
    scores = cross_val_score(classifier, X, y, cv=cvKFold, scoring='accuracy')
    return scores.mean()

nb_accuracy = nbClassifier(X, y)

### Part 1 Results


In [9]:
# Print results for each classifier in part 1 to 4 decimal places here:
print("LogR average cross-validation accuracy: {:.4f}".format(logreg_accuracy))
print("NB average cross-validation accuracy: {:.4f}".format(nb_accuracy))

LogR average cross-validation accuracy: 0.6700
NB average cross-validation accuracy: 0.6555


### Part 2: Cross-validation with parameter tuning

The data is first split into a training set and a test set with stratification and
`random_state=0`. Grid search with the same 10-fold stratified cross-validation
(`cvKFold`) is run **on the training set only**, and the best estimator found is then
evaluated once on the held-out test set.

In [10]:
# Split the pre-processed data into training and test subsets.
# Stratification preserves the class proportions in both subsets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)

print("Training examples:", X_train.shape[0])
print("Test examples:", X_test.shape[0])


def run_grid_search(classifier, param_grid, X_train, y_train, X_test, y_test):
    """Tune a classifier with grid search and evaluate it on the test set.

    Arguments:
        classifier: an unfitted sklearn estimator
        param_grid: dictionary of hyperparameter values to search over
        X_train, y_train: training data used for the grid search
        X_test, y_test: held-out data used for the final evaluation

    Returns:
        results: dictionary with the best parameters, the best cross-validation
                 accuracy, the test set accuracy and the macro / weighted F1 scores
    """
    grid_search = GridSearchCV(classifier, param_grid, cv=cvKFold,
                               scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    y_predicted = grid_search.best_estimator_.predict(X_test)

    results = {
        'best_params': grid_search.best_params_,
        'cv_accuracy': grid_search.best_score_,
        'test_accuracy': accuracy_score(y_test, y_predicted),
        'macro_f1': f1_score(y_test, y_predicted, average='macro'),
        'weighted_f1': f1_score(y_test, y_predicted, average='weighted'),
    }
    return results

Training examples: 167
Test examples: 42


In [11]:
# KNN 
# parameters may consider
k = [1, 3, 5, 7]
p = [1, 2]

knn_results = run_grid_search(
    KNeighborsClassifier(),
    {'n_neighbors': k, 'p': p},
    X_train, y_train, X_test, y_test)

In [12]:
# Decision Tree 
# parameters may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

# criterion='entropy' means the tree is grown using information gain, as in the tutorials.
dt_results = run_grid_search(
    DecisionTreeClassifier(criterion='entropy', random_state=0),
    {'max_depth': max_depth,
     'min_samples_split': min_samples_split,
     'min_samples_leaf': min_samples_leaf},
    X_train, y_train, X_test, y_test)

In [13]:
# Ada Boost
# parameters may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

ab_results = run_grid_search(
    AdaBoostClassifier(random_state=0),
    {'n_estimators': n_estimators, 'learning_rate': learning_rate},
    X_train, y_train, X_test, y_test)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/ana

In [14]:
# Gradient Boost
# parameters may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

gb_results = run_grid_search(
    GradientBoostingClassifier(random_state=0),
    {'max_depth': max_depth,
     'n_estimators': n_estimators,
     'learning_rate': learning_rate},
    X_train, y_train, X_test, y_test)

In [15]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to 'sqrt'.
# parameters may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

rf_results = run_grid_search(
    RandomForestClassifier(criterion='entropy', max_features='sqrt', random_state=0),
    {'n_estimators': n_estimators, 'max_leaf_nodes': max_leaf_nodes},
    X_train, y_train, X_test, y_test)

In [16]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = ['rbf']

svm_results = run_grid_search(
    SVC(random_state=0),
    {'C': C, 'gamma': gamma, 'kernel': kernel},
    X_train, y_train, X_test, y_test)

### Part 2: Results

In [17]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

print("KNN best k: {}".format(knn_results['best_params']['n_neighbors']))
print("KNN best p: {}".format(knn_results['best_params']['p']))
print("KNN cross-validation accuracy: {:.4f}".format(knn_results['cv_accuracy']))
print("KNN test set accuracy: {:.4f}".format(knn_results['test_accuracy']))
print()

print("DT best max_depth: {}".format(dt_results['best_params']['max_depth']))
print("DT best min_samples_split: {}".format(dt_results['best_params']['min_samples_split']))
print("DT best min_samples_leaf: {}".format(dt_results['best_params']['min_samples_leaf']))
print("DT cross-validation accuracy: {:.4f}".format(dt_results['cv_accuracy']))
print("DT test set accuracy: {:.4f}".format(dt_results['test_accuracy']))
print()

print("AdaBoost best n_estimators: {}".format(ab_results['best_params']['n_estimators']))
print("AdaBoost best learning_rate: {:.4f}".format(ab_results['best_params']['learning_rate']))
print("AdaBoost cross-validation accuracy: {:.4f}".format(ab_results['cv_accuracy']))
print("AdaBoost test set accuracy: {:.4f}".format(ab_results['test_accuracy']))
print()

print("GB best max_depth: {}".format(gb_results['best_params']['max_depth']))
print("GB best n_estimators: {}".format(gb_results['best_params']['n_estimators']))
print("GB best learning_rate: {:.4f}".format(gb_results['best_params']['learning_rate']))
print("GB cross-validation accuracy: {:.4f}".format(gb_results['cv_accuracy']))
print("GB test set accuracy: {:.4f}".format(gb_results['test_accuracy']))
print()

print("SVM best C: {:.4f}".format(svm_results['best_params']['C']))
print("SVM best gamma: {:.4f}".format(svm_results['best_params']['gamma']))
print("SVM cross-validation accuracy: {:.4f}".format(svm_results['cv_accuracy']))
print("SVM test set accuracy: {:.4f}".format(svm_results['test_accuracy']))
print()

print("RF best n_estimators: {}".format(rf_results['best_params']['n_estimators']))
print("RF best max_leaf_nodes: {}".format(rf_results['best_params']['max_leaf_nodes']))
print("RF cross-validation accuracy: {:.4f}".format(rf_results['cv_accuracy']))
print("RF test set accuracy: {:.4f}".format(rf_results['test_accuracy']))
print("RF test set macro average F1: {:.4f}".format(rf_results['macro_f1']))
print("RF test set weighted average F1: {:.4f}".format(rf_results['weighted_f1']))

KNN best k: 3
KNN best p: 1
KNN cross-validation accuracy: 0.7180
KNN test set accuracy: 0.5476

DT best max_depth: 5
DT best min_samples_split: 10
DT best min_samples_leaf: 1
DT cross-validation accuracy: 0.7824
DT test set accuracy: 0.6667

AdaBoost best n_estimators: 150
AdaBoost best learning_rate: 0.2000
AdaBoost cross-validation accuracy: 0.8077
AdaBoost test set accuracy: 0.6667

GB best max_depth: 1
GB best n_estimators: 100
GB best learning_rate: 0.3000
GB cross-validation accuracy: 0.8199
GB test set accuracy: 0.7143

SVM best C: 5.0000
SVM best gamma: 10.0000
SVM cross-validation accuracy: 0.6993
SVM test set accuracy: 0.5952

RF best n_estimators: 60
RF best max_leaf_nodes: 12
RF cross-validation accuracy: 0.8202
RF test set accuracy: 0.6667
RF test set macro average F1: 0.6541
RF test set weighted average F1: 0.6635


## **AI Acknowledgement**